# Creating a Simple Agent with Tracing

In [1]:
import dotenv
import os

from openai import OpenAI

dotenv.load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    print(
        """Error: OPENAI_API_KEY environment variable not set. Please copy the .env.template file as .env and fill it in.
    
    You can execute these commands in the terminal to get started:
    cp .env.template .env
    code .env
    """
    )

# Test OpenAI Access
print(
    OpenAI()
    .responses.create(
        model=os.environ["OPENAI_DEFAULT_MODEL"], input="Say: We are up and running!"
    )
    .output_text
)

We are up and running!


In [2]:
from agents import Agent, Runner, trace
from openai.types.responses import ResponseTextDeltaEvent

Create a simple Nutrition Assistant Agent

In [4]:
nutrition_agent = Agent(
    name = "Nutrition Assistant",
    instructions = """
    You are a helpful assistant giving out nutrition advice.
    You give concise answers.
    """,
)

teaching_agent = Agent(
    name = "Teaching Assistant",
    instructions = """
    You are a subject matter expert and will give out advice on Subject EVS - Environmental Studies.
    You give concise answers.
    """,
)

math_tutor_agent = Agent(
    name="Math Tutor",
    instructions="""
    You are a friendly, patient math tutor.
    Explain solutions step by step and use clear, age-appropriate language.
    Help students understand the reasoning instead of only giving the answer.
    If a problem is unclear, ask a brief clarifying question.
    """,
)


Let's execute the Agent:

In [5]:
with trace("Simple Teaching Agent for EVS"):
    result = await Runner.run(teaching_agent, "What is photosynthesis?")

print(result)

RunResult:
- Last agent: Agent(name="Teaching Assistant", ...)
- Final output (str):
    Photosynthesis is the process by which green plants (and some algae/bacteria) use light energy to convert carbon dioxide and water into glucose (a sugar) and oxygen. It occurs in chloroplasts.
    
    Overall equation:
    6 CO2 + 6 H2O + light energy → C6H12O6 + 6 O2
    
    Occurs in two main stages: light-dependent reactions and the Calvin cycle (dark reactions).
- 2 new item(s)
- 1 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)


Streaming the answer to the screen, token by token

In [6]:
response_stream = Runner.run_streamed(nutrition_agent, "Whats the nutritional value of apricots?")

async for event in response_stream.stream_events():
    if event.type == "raw_response_event" and isinstance(
        event.data, ResponseTextDeltaEvent
    ):
        print(event.data.delta, end="", flush=True)

Here’s a quick estimate for fresh apricots per 100 g:

- Calories: ~48
- Carbs: ~11 g (sugars ~9 g)
- Fiber: ~2 g
- Protein: ~1–1.5 g
- Fat: ~0.4 g
- Vitamin A: rich (beta-carotene)
- Vitamin C: present (smaller amount than citrus)
- Potassium: ~260 mg
- Other minerals in small amounts (calcium, iron, magnesium)

Dried apricots are much more energy-dense (about 240 kcal per 100 g) and higher in sugar. Values vary by variety and ripeness.

_Good Job!_